In [1]:

from pathlib import Path
import shutil
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

print("=" * 75)
print("SCR-CMX: DISK AND ARCHITECTURE INSPECTION")
print("=" * 75)

# 1. Disk space check
usage = shutil.disk_usage(SCR_DIR)
print("\n--- Disk usage ---")
print(f"Total:     {usage.total / 1024**3:.2f} GB")
print(f"Used:      {usage.used / 1024**3:.2f} GB")
print(f"Free:      {usage.free / 1024**3:.2f} GB")

if usage.free / 1024**3 < 20:
    print("WARNING: Less than 20 GB free. Avoid another full checkpoint copy.")

# 2. Check important files in the SCR copy
files_to_inspect = [
    "models/net_utils_e5.py",
    "models/encoders/dual_segformer_e5.py",
    "models/builder.py",
    "train.py",
    "config.py",
]

for relative_path in files_to_inspect:
    path = SCR_DIR / relative_path
    print(f"\n{'=' * 75}\nFILE: {relative_path}")
    if not path.exists():
        print("FILE MISSING")
        continue

    lines = path.read_text(errors="replace").splitlines()
    print(f"Total lines: {len(lines)}")

    # Locate architecture and loss-related definitions
    patterns = re.compile(
        r"^\s*(class |def |    def |    class )|"
        r"reliability|counterfactual|FeatureFusion|FeatureRectify|"
        r"backbone|optimizer|criterion|loss"
    )

    matches = [
        (i, line) for i, line in enumerate(lines)
        if patterns.search(line)
    ]

    # Print matches with nearby context, merging overlapping ranges
    ranges = []
    for i, _ in matches:
        start, end = max(0, i - 2), min(len(lines), i + 4)
        if ranges and start <= ranges[-1][1]:
            ranges[-1] = (ranges[-1][0], max(ranges[-1][1], end))
        else:
            ranges.append((start, end))

    # Limit output so the notebook does not become enormous
    printed = 0
    max_lines = 240 if "net_utils" in relative_path else 200

    for start, end in ranges:
        chunk_len = end - start
        if printed + chunk_len > max_lines:
            print("... output limit reached; remaining sections omitted ...")
            break

        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")
        printed += chunk_len

print("\nInspection complete. No source files were modified.")


SCR-CMX: DISK AND ARCHITECTURE INSPECTION

--- Disk usage ---
Total:     437.93 GB
Used:      360.62 GB
Free:      54.99 GB

FILE: models/net_utils_e5.py
Total lines: 78

--- Lines 2-19 ---
   2: import torch.nn as nn
   3: 
   4: from .net_utils import FeatureFusionModule
   5: 
   6: 
   7: class ReliabilityGatedFeatureRectifyModule(nn.Module):
   8:     """
   9:     E4 prototype:
  10:     Estimate per-pixel RGB/thermal reliability and use it to
  11:     gate cross-modal feature corrections.
  12: 
  13:     This module does not yet implement counterfactual supervision.
  14:     """
  15: 
  16:     def __init__(self, dim, reduction=1, lambda_c=0.5, lambda_s=0.5):
  17:         super().__init__()
  18: 
  19:         # Reuse the original CMX channel and spatial weighting mechanisms.

--- Lines 31-46 ---
  31: 
  32:         hidden = max(dim // 8, 8)
  33:         self.reliability_head = nn.Sequential(
  34:             nn.Conv2d(dim * 2, hidden, kernel_size=1),
  35:             

In [2]:

from pathlib import Path
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

checks = {
    "Config": SCR_DIR / "config.py",
    "Builder": SCR_DIR / "models/builder.py",
    "E5 encoder": SCR_DIR / "models/encoders/dual_segformer_e5.py",
    "E5 reliability module": SCR_DIR / "models/net_utils_e5.py",
    "E3 fusion module": SCR_DIR / "models/net_utils_e3.py",
    "E5 checkpoint": SCR_DIR / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth",
}

print("SCR-CMX workspace verification")
print("=" * 65)
for name, path in checks.items():
    print(f"{name:26} {'OK' if path.exists() else 'MISSING'}")

# Find where FRMs and FFMs are initialized in the encoder
encoder_path = checks["E5 encoder"]
if encoder_path.exists():
    lines = encoder_path.read_text(errors="replace").splitlines()
    print("\n--- Encoder initialization and stage fusion ---")
    patterns = re.compile(
        r"self\.FRMs|self\.FFMs|ReliabilityGuidedFusionModule|"
        r"FeatureFusionModule|def __init__|def forward_features"
    )
    matching = [i for i, line in enumerate(lines) if patterns.search(line)]
    shown = set()
    for i in matching:
        for j in range(max(0, i - 3), min(len(lines), i + 5)):
            if j not in shown:
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)

# Show dataset and backbone configuration
config_path = checks["Config"]
if config_path.exists():
    lines = config_path.read_text(errors="replace").splitlines()
    print("\n--- Relevant configuration ---")
    patterns = re.compile(
        r"C\.backbone|C\.dataset_path|C\.dataset_name|C\.num_classes|"
        r"C\.batch_size|C\.nepochs|C\.lr|C\.niters_per_epoch|"
        r"C\.pretrained_model|C\.log_dir"
    )
    for i, line in enumerate(lines):
        if patterns.search(line):
            print(f"{i + 1:4}: {line}")

print("\nVerification finished; no files changed and no training started.")


SCR-CMX workspace verification
Config                     OK
Builder                    OK
E5 encoder                 OK
E5 reliability module      OK
E3 fusion module           OK
E5 checkpoint              OK

--- Encoder initialization and stage fusion ---
   4: from functools import partial
   5: 
   6: from timm.models.layers import DropPath, to_2tuple, trunc_normal_
   7: from ..net_utils_e3 import ReliabilityGuidedFusionModule as FFM
   8: from ..net_utils_e5 import ReliabilityGatedFeatureRectifyModule as FRM
   9: import math
  10: import time
  11: from engine.logger import get_logger
  17:     """
  18:     Depthwise convolution bloc: input: x with size(B N C); output size (B N C)
  19:     """
  20:     def __init__(self, dim=768):
  21:         super(DWConv, self).__init__()
  22:         self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, bias=True, groups=dim)
  23: 
  24:     def forward(self, x, H, W):
  31: 
  32: 
  33: class Mlp(nn.Module):
  34:   

In [3]:

from pathlib import Path
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files_to_inspect = {
    "E3 fusion module": SCR_DIR / "models/net_utils_e3.py",
    "E5 encoder": SCR_DIR / "models/encoders/dual_segformer_e5.py",
    "E5 builder": SCR_DIR / "models/builder.py",
}

patterns = {
    "E3 fusion module": re.compile(
        r"class ReliabilityGuidedFusionModule|"
        r"def __init__|def forward|self\.|return "
    ),
    "E5 encoder": re.compile(
        r"self\.FRMs\s*=|self\.FFMs\s*=|"
        r"def forward_features|self\.last_reliabilities|"
        r"self\.FRMs\[|self\.FFMs\["
    ),
    "E5 builder": re.compile(
        r"class EncoderDecoder|def __init__|"
        r"def encode_decode|def _e5_counterfactual_loss|"
        r"def forward|mit_b2_e5|last_reliabilities"
    ),
}

for name, path in files_to_inspect.items():
    print("\n" + "=" * 90)
    print(name, "|", "EXISTS:", path.exists())
    print("=" * 90)

    if not path.exists():
        continue

    lines = path.read_text(errors="replace").splitlines()
    selected = set()

    for i, line in enumerate(lines):
        if patterns[name].search(line):
            selected.update(
                range(max(0, i - 2), min(len(lines), i + 4))
            )

    previous = -2
    for i in sorted(selected):
        if i > previous + 1:
            print("    ...")
        print(f"{i + 1:4}: {lines[i]}")
        previous = i

print("\nInspection complete. No source files modified; no training started.")



E3 fusion module | EXISTS: True
    ...
   5: 
   6: 
   7: class ReliabilityGuidedFusionModule(FeatureFusionModule):
   8:     """
   9:     E3 prototype:
  10:     Preserve CMX CrossPath and ChannelEmbed, but estimate spatially
    ...
  12:     """
  13: 
  14:     def __init__(self, dim, reduction=1, num_heads=None,
  15:                  norm_layer=nn.BatchNorm2d):
  16:         super().__init__(
  17:             dim=dim,
    ...
  22: 
  23:         hidden = max(dim // 8, 8)
  24:         self.reliability_head = nn.Sequential(
  25:             nn.Conv2d(dim * 2, hidden, kernel_size=1),
  26:             nn.ReLU(inplace=True),
  27:             nn.Conv2d(hidden, 2, kernel_size=1),
  28:         )
  29: 
  30:     def forward(self, x1, x2):
  31:         B, C, H, W = x1.shape
  32: 
  33:         # Original CMX cross-modal interaction.
  34:         t1 = x1.flatten(2).transpose(1, 2)
  35:         t2 = x2.flatten(2).transpose(1, 2)
  36:         t1, t2 = self.cross(t1, t2)
  37:

In [4]:

from pathlib import Path

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

targets = [
    ("E3 reliability-guided fusion",
     SCR_DIR / "models/net_utils_e3.py",
     "class ReliabilityGuidedFusionModule"),
    ("Original CMX fusion base",
     SCR_DIR / "models/net_utils.py",
     "class FeatureFusionModule"),
]

for title, path, marker in targets:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    if not path.exists():
        print("FILE MISSING:", path)
        continue

    lines = path.read_text(errors="replace").splitlines()
    matches = [
        i for i, line in enumerate(lines)
        if marker in line
    ]

    if not matches:
        print("Class not found:", marker)
        continue

    for start in matches:
        # Print the complete class until the next top-level class.
        end = len(lines)
        for i in range(start + 1, len(lines)):
            if lines[i].startswith("class "):
                end = i
                break

        print(f"\nLines {start + 1}-{end}:")
        for i in range(start, end):
            print(f"{i + 1:4}: {lines[i]}")

print("\nInspection complete. No files modified.")



E3 reliability-guided fusion

Lines 7-53:
   7: class ReliabilityGuidedFusionModule(FeatureFusionModule):
   8:     """
   9:     E3 prototype:
  10:     Preserve CMX CrossPath and ChannelEmbed, but estimate spatially
  11:     varying RGB/thermal fusion weights after cross-modal interaction.
  12:     """
  13: 
  14:     def __init__(self, dim, reduction=1, num_heads=None,
  15:                  norm_layer=nn.BatchNorm2d):
  16:         super().__init__(
  17:             dim=dim,
  18:             reduction=reduction,
  19:             num_heads=num_heads,
  20:             norm_layer=norm_layer,
  21:         )
  22: 
  23:         hidden = max(dim // 8, 8)
  24:         self.reliability_head = nn.Sequential(
  25:             nn.Conv2d(dim * 2, hidden, kernel_size=1),
  26:             nn.ReLU(inplace=True),
  27:             nn.Conv2d(hidden, 2, kernel_size=1),
  28:         )
  29: 
  30:     def forward(self, x1, x2):
  31:         B, C, H, W = x1.shape
  32: 
  33:         # 

In [5]:

from pathlib import Path
import shutil

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

fusion_file = SCR_DIR / "models/net_utils_scr.py"
encoder_file = SCR_DIR / "models/encoders/dual_segformer_e5.py"
backup_file = SCR_DIR / "models/encoders/dual_segformer_e5.py.before_scr"

assert fusion_file.parent.exists(), "SCR models directory not found"
assert encoder_file.exists(), "SCR E5 encoder not found"
assert not fusion_file.exists(), (
    "SCR fusion file already exists. Stop and inspect it before overwriting."
)

# Preserve the original SCR copy of the encoder before editing.
if not backup_file.exists():
    shutil.copy2(encoder_file, backup_file)

fusion_source = r'''
import torch
import torch.nn as nn

from .net_utils import FeatureFusionModule


class SpatialReliabilityConditionedFusion(FeatureFusionModule):
    """
    SCR-CMX: Spatial Reliability-Conditioned Cross-Modal Interaction.

    Reuses the CMX CrossPath and ChannelEmbed modules.
    Reliability controls the cross-modal residual update rather than
    weighting the completed cross-modal features after interaction.
    """

    def __init__(
        self,
        dim,
        reduction=1,
        num_heads=None,
        norm_layer=nn.BatchNorm2d,
    ):
        super().__init__(
            dim=dim,
            reduction=reduction,
            num_heads=num_heads,
            norm_layer=norm_layer,
        )

    def forward(self, x_rgb, x_thermal, reliability):
        if x_rgb.shape != x_thermal.shape:
            raise ValueError(
                f"RGB and thermal feature shapes differ: "
                f"{x_rgb.shape} vs {x_thermal.shape}"
            )

        if reliability.ndim != 4 or reliability.shape[1] != 2:
            raise ValueError(
                "Expected reliability shape (B, 2, H, W), "
                f"received {tuple(reliability.shape)}"
            )

        B, C, H, W = x_rgb.shape

        if reliability.shape[0] != B or reliability.shape[2:] != (H, W):
            raise ValueError(
                "Reliability batch/spatial dimensions must match features: "
                f"{tuple(reliability.shape)} vs {tuple(x_rgb.shape)}"
            )

        r_rgb = reliability[:, 0:1].clamp(0.0, 1.0)
        r_thermal = reliability[:, 1:2].clamp(0.0, 1.0)

        # Original features as token sequences.
        rgb_tokens = x_rgb.flatten(2).transpose(1, 2)
        thermal_tokens = x_thermal.flatten(2).transpose(1, 2)

        # Existing CMX cross-modal interaction.
        rgb_cross, thermal_cross = self.cross(rgb_tokens, thermal_tokens)

        # Return to spatial feature maps.
        rgb_cross = (
            rgb_cross.transpose(1, 2)
            .reshape(B, C, H, W)
            .contiguous()
        )
        thermal_cross = (
            thermal_cross.transpose(1, 2)
            .reshape(B, C, H, W)
            .contiguous()
        )

        # Interaction deltas: isolate the changes introduced by CrossPath.
        rgb_delta = rgb_cross - x_rgb
        thermal_delta = thermal_cross - x_thermal

        # Reliability-conditioned cross-modal updates.
        # RGB receives thermal information according to thermal reliability.
        # Thermal receives RGB information according to RGB reliability.
        rgb_out = x_rgb + r_thermal * rgb_delta
        thermal_out = x_thermal + r_rgb * thermal_delta

        # Keep the original CMX channel embedding interface.
        rgb_tokens = rgb_out.flatten(2).transpose(1, 2)
        thermal_tokens = thermal_out.flatten(2).transpose(1, 2)
        merged = torch.cat([rgb_tokens, thermal_tokens], dim=-1)

        return self.channel_emb(merged, H, W)
'''

fusion_file.write_text(fusion_source.strip() + "\n")

print("Created:", fusion_file)
print("Encoder backup:", backup_file)
print("Original E0-E5 workspaces were not modified.")
print("No training started.")


Created: /home/cse-sdpl/Downloads/CMX_SCR/models/net_utils_scr.py
Encoder backup: /home/cse-sdpl/Downloads/CMX_SCR/models/encoders/dual_segformer_e5.py.before_scr
Original E0-E5 workspaces were not modified.
No training started.


In [6]:

from pathlib import Path
import ast
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files = {
    "SCR fusion": SCR_DIR / "models/net_utils_scr.py",
    "CMX base modules": SCR_DIR / "models/net_utils.py",
    "E5 encoder": SCR_DIR / "models/encoders/dual_segformer_e5.py",
}

for name, path in files.items():
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

# Show CrossPath and ChannelEmbed definitions with line numbers.
base_file = files["CMX base modules"]
lines = base_file.read_text(errors="replace").splitlines()

for class_name in ("class CrossPath", "class ChannelEmbed"):
    starts = [i for i, line in enumerate(lines) if line.startswith(class_name)]
    for start in starts:
        end = len(lines)
        for i in range(start + 1, len(lines)):
            if lines[i].startswith("class "):
                end = i
                break
        print(f"\n--- {class_name} (lines {start+1}-{end}) ---")
        for i in range(start, end):
            print(f"{i+1:4}: {lines[i]}")

# Inspect the encoder's existing reliability and fusion calls.
encoder_file = files["E5 encoder"]
enc_lines = encoder_file.read_text(errors="replace").splitlines()

print("\n--- Encoder FRM/FFM setup and calls ---")
pattern = re.compile(r"self\.FRMs\s*=|self\.FFMs\s*=|self\.FRMs\[|self\.FFMs\[")
for i, line in enumerate(enc_lines):
    if pattern.search(line):
        lo, hi = max(0, i-2), min(len(enc_lines), i+4)
        for j in range(lo, hi):
            print(f"{j+1:4}: {enc_lines[j]}")
        print()

print("\nInspection only: no files modified and no training started.")


SCR fusion: OK
CMX base modules: OK
E5 encoder: OK

--- class CrossPath (lines 111-135) ---
 111: class CrossPath(nn.Module):
 112:     def __init__(self, dim, reduction=1, num_heads=None, norm_layer=nn.LayerNorm):
 113:         super().__init__()
 114:         self.channel_proj1 = nn.Linear(dim, dim // reduction * 2)
 115:         self.channel_proj2 = nn.Linear(dim, dim // reduction * 2)
 116:         self.act1 = nn.ReLU(inplace=True)
 117:         self.act2 = nn.ReLU(inplace=True)
 118:         self.cross_attn = CrossAttention(dim // reduction, num_heads=num_heads)
 119:         self.end_proj1 = nn.Linear(dim // reduction * 2, dim)
 120:         self.end_proj2 = nn.Linear(dim // reduction * 2, dim)
 121:         self.norm1 = norm_layer(dim)
 122:         self.norm2 = norm_layer(dim)
 123: 
 124:     def forward(self, x1, x2):
 125:         y1, u1 = self.act1(self.channel_proj1(x1)).chunk(2, dim=-1)
 126:         y2, u2 = self.act2(self.channel_proj2(x2)).chunk(2, dim=-1)
 127:       

In [7]:

from pathlib import Path
import re
import py_compile

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
encoder_path = SCR_DIR / "models/encoders/dual_segformer_e5.py"
fusion_path = SCR_DIR / "models/net_utils_scr.py"
backup_path = SCR_DIR / "models/encoders/dual_segformer_e5.py.before_scr"

assert encoder_path.exists(), f"Missing encoder: {encoder_path}"
assert fusion_path.exists(), f"Missing SCR fusion module: {fusion_path}"
assert backup_path.exists(), f"Missing original encoder backup: {backup_path}"

source = encoder_path.read_text()

# Avoid accidentally applying the patch twice.
if "SCR-CMX: reliability-conditioned fusion" in source:
    print("SCR encoder patch already present; no changes made.")
else:
    # Import the new SCR fusion module.
    import_line = (
        "from ..net_utils_e5 import ReliabilityGatedFeatureRectifyModule as FRM"
    )
    scr_import = (
        import_line
        + "\nfrom ..net_utils_scr import SpatialReliabilityConditionedFusion as SCRFFM"
    )
    assert import_line in source, "Could not find the expected E5 FRM import."
    source = source.replace(import_line, scr_import, 1)

    # Replace the four fusion modules with SCR fusion modules.
    old_ffm_block = re.compile(
        r"        self\.FFMs = nn\.ModuleList\(\[\s*"
        r"FFM\(dim=embed_dims\[0\].*?\n\s*"
        r"FFM\(dim=embed_dims\[3\].*?\]\)",
        re.DOTALL,
    )

    replacement = '''        # SCR-CMX: reliability-conditioned fusion at all four stages.
        self.FFMs = nn.ModuleList([
            SCRFFM(dim=embed_dims[0], reduction=1, num_heads=num_heads[0], norm_layer=norm_fuse),
            SCRFFM(dim=embed_dims[1], reduction=1, num_heads=num_heads[1], norm_layer=norm_fuse),
            SCRFFM(dim=embed_dims[2], reduction=1, num_heads=num_heads[2], norm_layer=norm_fuse),
            SCRFFM(dim=embed_dims[3], reduction=1, num_heads=num_heads[3], norm_layer=norm_fuse),
        ])'''

    source, count = old_ffm_block.subn(replacement, source, count=1)
    assert count == 1, (
        "Could not safely locate the original four-stage FFM block. "
        "Encoder was not written."
    )

    # Pass each existing E5 reliability map to SCR fusion.
    for stage in range(4):
        old_call = f"x_fused = self.FFMs[{stage}](x_rgb, x_e)"
        new_call = (
            f"# SCR-CMX: reliability-conditioned fusion\n"
            f"        x_fused = self.FFMs[{stage}](x_rgb, x_e, reliability)"
        )
        assert old_call in source, f"Missing expected fusion call for stage {stage + 1}"
        source = source.replace(old_call, new_call, 1)

    # Validate Python syntax before writing the source.
    compile(source, str(encoder_path), "exec")

    encoder_path.write_text(source)
    print("SCR encoder patched successfully.")

# Compile both files without importing or training the model.
py_compile.compile(str(encoder_path), doraise=True)
py_compile.compile(str(fusion_path), doraise=True)

print("Encoder syntax: OK")
print("SCR fusion syntax: OK")
print("Original E5 encoder backup remains:", backup_path)
print("No training started.")


SCR encoder patched successfully.
Encoder syntax: OK
SCR fusion syntax: OK
Original E5 encoder backup remains: /home/cse-sdpl/Downloads/CMX_SCR/models/encoders/dual_segformer_e5.py.before_scr
No training started.


In [8]:

from pathlib import Path
import sys
import torch
import traceback

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = (
    SCR_DIR / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
)

assert SCR_DIR.exists(), f"SCR directory missing: {SCR_DIR}"
assert E5_CHECKPOINT.exists(), f"E5 checkpoint missing: {E5_CHECKPOINT}"

# Run imports from the SCR workspace.
sys.path.insert(0, str(SCR_DIR))
import os
os.chdir(SCR_DIR)

import config
from models.builder import EncoderDecoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Match the E5 configuration but avoid creating a training run.
cfg = config.C
cfg.backbone = "mit_b2_e5"
cfg.num_classes = 9

model = EncoderDecoder(cfg=cfg).to(device)
print("Model constructed.")

# Inspect the checkpoint structure.
checkpoint = torch.load(
    str(E5_CHECKPOINT),
    map_location="cpu",
    weights_only=False,
)

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys())[:20])

    if "model" in checkpoint:
        state_dict = checkpoint["model"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
    else:
        # Some training scripts save the state dictionary directly.
        state_dict = checkpoint
else:
    raise TypeError(f"Unexpected checkpoint type: {type(checkpoint)}")

# Load only shape-compatible keys; report exactly what was skipped.
current_state = model.state_dict()
compatible = {
    key: value
    for key, value in state_dict.items()
    if key in current_state
    and hasattr(value, "shape")
    and current_state[key].shape == value.shape
}

print("Checkpoint tensors:", len(state_dict))
print("Shape-compatible tensors:", len(compatible))
print("Model tensors:", len(current_state))

load_result = model.load_state_dict(compatible, strict=False)
print("Missing model keys:", len(load_result.missing_keys))
print("Unexpected checkpoint keys:", len(load_result.unexpected_keys))
print("Sample missing keys:", load_result.missing_keys[:15])

# Synthetic forward/backward test at 640 x 480.
model.train()
model.zero_grad(set_to_none=True)

rgb = torch.randn(1, 3, 480, 640, device=device)
thermal = torch.randn(1, 3, 480, 640, device=device)
labels = torch.randint(0, 9, (1, 480, 640), device=device)

try:
    with torch.enable_grad():
        loss = model(rgb, thermal, labels)

        if isinstance(loss, (tuple, list)):
            loss = sum(x for x in loss if torch.is_tensor(x) and x.numel() == 1)

        if not torch.is_tensor(loss) or loss.numel() != 1:
            raise RuntimeError(
                f"Expected scalar training loss, got {type(loss)} "
                f"with shape {getattr(loss, 'shape', None)}"
            )

        print("Loss:", float(loss.detach().cpu()))
        print("Loss finite:", bool(torch.isfinite(loss).all()))

        loss.backward()

    grads = [
        p.grad for p in model.parameters()
        if p.requires_grad and p.grad is not None
    ]
    finite_grads = all(bool(torch.isfinite(g).all()) for g in grads)

    print("Parameters with gradients:", len(grads))
    print("All observed gradients finite:", finite_grads)

    scr_parameters = [
        (name, p) for name, p in model.named_parameters()
        if "FFMs" in name and p.requires_grad
    ]
    scr_grad_count = sum(p.grad is not None for _, p in scr_parameters)

    print("Trainable SCR fusion parameters:", len(scr_parameters))
    print("SCR fusion parameters receiving gradients:", scr_grad_count)

    if not bool(torch.isfinite(loss).all()) or not finite_grads:
        raise RuntimeError("Non-finite loss or gradients detected.")

    print("\nSMOKE TEST PASSED")

except Exception:
    print("\nSMOKE TEST FAILED")
    traceback.print_exc()
    raise

finally:
    del rgb, thermal, labels
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("No optimizer step was performed; no training checkpoint was saved.")


23 17:56:42 Using E5: SegFormer-B2 with counterfactual reliability


Device: cuda


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
23 17:56:44 Using MLP Decoder
23 17:56:44 Loading pretrained model: /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
23 17:56:44 Load model, Time usage:
	IO: 0.05199480056762695, initialize parameters: 0.027271032333374023
23 17:56:44 Initing weights ...


Model constructed.
Checkpoint keys: ['model', 'optimizer', 'epoch', 'iteration']
Checkpoint tensors: 869
Shape-compatible tensors: 853
Model tensors: 853
Missing model keys: 0
Unexpected checkpoint keys: 0
Sample missing keys: []
Loss: 18.851318359375
Loss finite: True
Parameters with gradients: 826
All observed gradients finite: True
Trainable SCR fusion parameters: 100
SCR fusion parameters receiving gradients: 100

SMOKE TEST PASSED
No optimizer step was performed; no training checkpoint was saved.


In [9]:

from pathlib import Path

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files_to_check = [
    SCR_DIR / "train.py",
    SCR_DIR / "models/builder.py",
    SCR_DIR / "models/encoders/dual_segformer_e5.py",
    SCR_DIR / "models/net_utils_scr.py",
]

keywords = [
    "load_state_dict",
    "optimizer",
    "params_list",
    "counterfactual",
    "last_reliabilities",
    "loss.backward",
    "optimizer.step",
    "torch.save",
]

for path in files_to_check:
    print("\n" + "=" * 90)
    print("FILE:", path)
    print("=" * 90)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(errors="replace").splitlines()
    matches = [
        i for i, line in enumerate(lines)
        if any(key.lower() in line.lower() for key in keywords)
    ]

    shown = set()
    for i in matches:
        start = max(0, i - 3)
        end = min(len(lines), i + 5)

        if any(j in shown for j in range(start, end)):
            continue

        print(f"\n--- lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")
            shown.add(j)



FILE: /home/cse-sdpl/Downloads/CMX_SCR/train.py

--- lines 59-66 ---
  59:     model=segmodel(cfg=config, criterion=criterion, norm_layer=BatchNorm2d)
  60: 
  61:     # E3: initialize compatible model weights from E2 only.
  62:     # Do not restore E2 optimizer, epoch, or training state.
  63:     if config.backbone == 'mit_b2_e3':
  64:         import torch
  65: 
  66:         e2_checkpoint_path = r"/home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth"

--- lines 77-84 ---
  77:             if key in e3_state and e3_state[key].shape == value.shape
  78:         }
  79: 
  80:         result = model.load_state_dict(compatible_state, strict=False)
  81: 
  82:         # Only E3 reliability-head parameters should be missing.
  83:         assert not result.unexpected_keys, result.unexpected_keys
  84:         assert all(

--- lines 100-107 ---
 100:     
 101: 
 102:     # E4: initialize compatible model weights from E3 only.
 103:     # Do not restore the E3 op

In [10]:

from pathlib import Path

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files_to_inspect = {
    "config.py": SCR_DIR / "config.py",
    "builder.py": SCR_DIR / "models" / "builder.py",
    "train.py": SCR_DIR / "train.py",
}

keywords = [
    "backbone",
    "log_path",
    "log_dir",
    "checkpoint",
    "snapshot",
    "save",
    "continue_state_object",
    "mit_b2_e5",
    "e5_counterfactual",
    "e5_reliability_loss_weight",
    "e5_corruption_fraction",
    "e5_counterfactual_temperature",
]

for label, path in files_to_inspect.items():
    print("\n" + "=" * 90)
    print(label, "| exists:", path.exists())
    print("=" * 90)

    if not path.exists():
        continue

    lines = path.read_text(errors="replace").splitlines()
    matched = set()

    for i, line in enumerate(lines):
        if any(key.lower() in line.lower() for key in keywords):
            start = max(0, i - 2)
            end = min(len(lines), i + 4)

            if any(j in matched for j in range(start, end)):
                continue

            print(f"\n--- lines {start + 1}-{end} ---")
            for j in range(start, end):
                print(f"{j + 1:4}: {lines[j]}")
                matched.add(j)



config.py | exists: True

--- lines 79-84 ---
  79: # ============================================================
  80: # Network Configuration
  81: C.backbone = 'mit_b2_e5'
  82: 
  83: C.pretrained_model = '/home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth'
  84: 

--- lines 118-123 ---
 118: 
 119: # ============================================================
 120: # CHECKPOINT CONFIGURATION
 121: # ============================================================
 122: C.checkpoint_start_epoch = 1
 123: C.checkpoint_step = 5

--- lines 135-140 ---
 135: # LOGGING CONFIGURATION
 136: # ============================================================
 137: C.log_dir = osp.abspath(
 138:     'log_' + C.dataset_name + '_' + C.backbone
 139: )
 140: 

--- lines 141-146 ---
 141: C.tb_dir = osp.join(C.log_dir, 'tb')
 142: C.log_dir_link = C.log_dir
 143: C.checkpoint_dir = osp.join(C.log_dir, 'checkpoint')
 144: 
 145: exp_time = time.strftime('%Y_%m_%d_%H_%M_%S', time.loc

In [11]:

from pathlib import Path
import shutil
import py_compile

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

config_path = SCR_DIR / "config.py"
builder_path = SCR_DIR / "models" / "builder.py"
train_path = SCR_DIR / "train.py"

# Read and validate before making any changes.
config_text = config_path.read_text()
builder_text = builder_path.read_text()
train_text = train_path.read_text()

assert "C.backbone = 'mit_b2_e5'" in config_text, \
    "Unexpected config.py backbone line; no changes made."

assert "elif cfg.backbone == 'mit_b2_e5':" in builder_text, \
    "Expected E5 builder branch not found; no changes made."

assert 'if config.backbone == \'mit_b2_e5\':' in train_text, \
    "Expected E5 initialization branch not found; no changes made."

# Back up the three files once.
for path in (config_path, builder_path, train_path):
    backup = path.with_name(path.name + ".before_scr_identity")
    if not backup.exists():
        shutil.copy2(path, backup)
    print("Backup:", backup)

# 1. Give SCR its own backbone name.
config_text = config_text.replace(
    "C.backbone = 'mit_b2_e5'",
    "C.backbone = 'mit_b2_scr'",
    1,
)

# 2. Add SCR builder branch before the E5 branch.
scr_branch = """        elif cfg.backbone == 'mit_b2_scr':
            logger.info('Using SCR-CMX: Spatial Reliability-Conditioned Fusion')
            from .encoders.dual_segformer_e5 import mit_b2 as backbone
            self.backbone = backbone(norm_fuse=norm_layer)
"""

builder_text = builder_text.replace(
    "        elif cfg.backbone == 'mit_b2_e5':",
    scr_branch + "        elif cfg.backbone == 'mit_b2_e5':",
    1,
)

# 3. Allow the counterfactual reliability loss to be explicitly enabled
# for SCR without changing E5's existing behavior.
old_flag = '''        self.e5_counterfactual = (
            getattr(cfg, "backbone", "") == "mit_b2_e5"
        )'''

new_flag = '''        self.e5_counterfactual = (
            getattr(cfg, "backbone", "") == "mit_b2_e5"
            or (
                getattr(cfg, "backbone", "") == "mit_b2_scr"
                and getattr(cfg, "scr_use_counterfactual_loss", True)
            )
        )'''

assert old_flag in builder_text, \
    "Counterfactual flag pattern not found; no changes written."

builder_text = builder_text.replace(old_flag, new_flag, 1)

# 4. Add a SCR-specific E5 initialization branch in train.py.
# It loads model weights only; the optimizer and schedule remain fresh.
scr_init = '''    # SCR: initialize compatible model weights from trained E5.
    # Do not restore E5 optimizer, epoch, or training state.
    if config.backbone == 'mit_b2_scr':
        import torch

        scr_init_path = (
            "/home/cse-sdpl/Downloads/CMX_SCR/"
            "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
        )
        if not __import__("os").path.isfile(scr_init_path):
            raise FileNotFoundError(
                f"SCR initialization checkpoint not found: {scr_init_path}"
            )

        scr_checkpoint = torch.load(
            scr_init_path,
            map_location="cpu",
            weights_only=False,
        )
        scr_state = (
            scr_checkpoint["model"]
            if isinstance(scr_checkpoint, dict)
            and "model" in scr_checkpoint
            else scr_checkpoint
        )
        current_state = model.state_dict()
        compatible_state = {
            key: value
            for key, value in scr_state.items()
            if key in current_state
            and current_state[key].shape == value.shape
        }

        result = model.load_state_dict(compatible_state, strict=False)
        print("SCR initialized from E5:", scr_init_path)
        print("Compatible checkpoint tensors:", len(compatible_state))
        print("Missing model keys:", len(result.missing_keys))
        print("Unexpected checkpoint keys:", len(result.unexpected_keys))
        print("Sample missing keys:", result.missing_keys[:20])

        assert not result.unexpected_keys, result.unexpected_keys

'''

train_text = train_text.replace(
    "    # E5: initialize model weights from the trained E4 checkpoint.",
    scr_init + "    # E5: initialize model weights from the trained E4 checkpoint.",
    1,
)

# 5. Write only after all expected patterns have been validated.
config_path.write_text(config_text)
builder_path.write_text(builder_text)
train_path.write_text(train_text)

# Syntax check all modified Python files.
for path in (config_path, builder_path, train_path):
    py_compile.compile(str(path), doraise=True)
    print("Syntax OK:", path.name)

print("\nSCR identity patch applied.")
print("Backbone:", "mit_b2_scr")
print("Counterfactual reliability loss: enabled by default for SCR")
print("Training started: NO")


Backup: /home/cse-sdpl/Downloads/CMX_SCR/config.py.before_scr_identity
Backup: /home/cse-sdpl/Downloads/CMX_SCR/models/builder.py.before_scr_identity
Backup: /home/cse-sdpl/Downloads/CMX_SCR/train.py.before_scr_identity
Syntax OK: config.py
Syntax OK: builder.py
Syntax OK: train.py

SCR identity patch applied.
Backbone: mit_b2_scr
Counterfactual reliability loss: enabled by default for SCR
Training started: NO


In [12]:

from pathlib import Path

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

config_text = (SCR_DIR / "config.py").read_text()
builder_text = (SCR_DIR / "models" / "builder.py").read_text()
train_text = (SCR_DIR / "train.py").read_text()

checks = {
    "SCR backbone selected in config":
        "C.backbone = 'mit_b2_scr'" in config_text,

    "Separate SCR builder branch":
        "elif cfg.backbone == 'mit_b2_scr':" in builder_text,

    "SCR-specific checkpoint initialization":
        "if config.backbone == 'mit_b2_scr':" in train_text,

    "E5 branch retained":
        "if config.backbone == 'mit_b2_e5':" in train_text,

    "SCR-specific log directory will be derived from backbone":
        "C.log_dir = osp.abspath(" in config_text
        and "C.backbone" in config_text,

    "SCR initialization checkpoint path present":
        "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth" in train_text,
}

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)

assert all(checks.values()), "One or more SCR patch checks failed."

print("\nVerification passed. No training has been started.")
print("Expected SCR log directory: log_TDD_mit_b2_scr")


PASS - SCR backbone selected in config
PASS - Separate SCR builder branch
PASS - SCR-specific checkpoint initialization
PASS - E5 branch retained
PASS - SCR-specific log directory will be derived from backbone
PASS - SCR initialization checkpoint path present

Verification passed. No training has been started.
Expected SCR log directory: log_TDD_mit_b2_scr


In [13]:

from pathlib import Path
import py_compile
import shutil

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
train_path = SCR_DIR / "train.py"
text = train_path.read_text()

old = "if config.backbone in ('mit_b2_e3', 'mit_b2_e4'):"
new = "if config.backbone in ('mit_b2_e3', 'mit_b2_e4', 'mit_b2_scr'):"

if new in text:
    print("SCR optimizer audit is already enabled.")
elif old in text:
    backup = train_path.with_name("train.py.before_scr_optimizer_audit")
    if not backup.exists():
        shutil.copy2(train_path, backup)
    text = text.replace(old, new, 1)
    train_path.write_text(text)
    py_compile.compile(str(train_path), doraise=True)
    print("SCR optimizer audit added successfully.")
    print("Backup:", backup)
else:
    raise RuntimeError(
        "Expected optimizer audit condition not found. No changes made."
    )

# Verify the final condition.
text = train_path.read_text()
assert new in text
print("Verified: SCR is included in the optimizer audit.")
print("Training started: NO")


SCR optimizer audit added successfully.
Backup: /home/cse-sdpl/Downloads/CMX_SCR/train.py.before_scr_optimizer_audit
Verified: SCR is included in the optimizer audit.
Training started: NO


In [14]:

from pathlib import Path
import os
import sys

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = (
    SCR_DIR / "log_TDD_mit_b2_e5" / "checkpoint" / "epoch-100.pth"
)

os.chdir(SCR_DIR)
if str(SCR_DIR) not in sys.path:
    sys.path.insert(0, str(SCR_DIR))

import config

cfg = config.C

print("Backbone:", cfg.backbone)
print("Log directory:", cfg.log_dir)
print("Checkpoint directory:", cfg.checkpoint_dir)
print("Dataset path:", cfg.dataset_path)
print("E5 initialization checkpoint exists:", E5_CHECKPOINT.is_file())
print("SCR counterfactual loss setting:",
      getattr(cfg, "scr_use_counterfactual_loss", True))
print("Counterfactual loss weight:",
      getattr(cfg, "e5_reliability_loss_weight", 0.1))

assert cfg.backbone == "mit_b2_scr"
assert Path(cfg.log_dir).name == "log_TDD_mit_b2_scr"
assert Path(cfg.checkpoint_dir).parent == Path(cfg.log_dir)
assert E5_CHECKPOINT.is_file(), (
    f"Missing initialization checkpoint: {E5_CHECKPOINT}"
)

print("\nPREFLIGHT PASSED")
print("No training started and no checkpoint saved.")


Backbone: mit_b2_e5
Log directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5
Checkpoint directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5/checkpoint
Dataset path: /home/cse-sdpl/Downloads/CMX_Original/datasets/TDD
E5 initialization checkpoint exists: True
SCR counterfactual loss setting: True
Counterfactual loss weight: 0.1


AssertionError: 

In [15]:

from pathlib import Path
import os
import sys
import importlib

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
os.chdir(SCR_DIR)

if str(SCR_DIR) not in sys.path:
    sys.path.insert(0, str(SCR_DIR))

# Remove the cached module before importing the SCR configuration.
if "config" in sys.modules:
    del sys.modules["config"]

import config
importlib.reload(config)

cfg = config.C

print("Loaded config file:", Path(config.__file__).resolve())
print("Backbone:", cfg.backbone)
print("Log directory:", cfg.log_dir)
print("Checkpoint directory:", cfg.checkpoint_dir)

print("\nConfig source lines:")
for i, line in enumerate(
    (SCR_DIR / "config.py").read_text().splitlines(), start=1
):
    if "C.backbone" in line or "C.log_dir =" in line:
        print(f"{i}: {line}")


Loaded config file: /home/cse-sdpl/Downloads/CMX_SCR/config.py
Backbone: mit_b2_scr
Log directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr
Checkpoint directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint

Config source lines:
81: C.backbone = 'mit_b2_scr'
137: C.log_dir = osp.abspath(
138:     'log_' + C.dataset_name + '_' + C.backbone
163:     print('Backbone:', C.backbone)


In [16]:

from pathlib import Path
import os
import sys
import torch

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = (
    SCR_DIR / "log_TDD_mit_b2_e5" / "checkpoint" / "epoch-100.pth"
)

os.chdir(SCR_DIR)
if str(SCR_DIR) not in sys.path:
    sys.path.insert(0, str(SCR_DIR))

import config
from models.builder import EncoderDecoder

cfg = config.C

assert cfg.backbone == "mit_b2_scr", cfg.backbone
assert E5_CHECKPOINT.is_file(), f"Missing checkpoint: {E5_CHECKPOINT}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Backbone:", cfg.backbone)
print("SCR output directory:", cfg.log_dir)

# Construct the SCR model. No training or checkpoint saving.
model = EncoderDecoder(cfg=cfg).to(device)
print("SCR model constructed.")

# Load compatible E5 weights only; do not restore optimizer or epoch.
checkpoint = torch.load(
    E5_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)
if isinstance(checkpoint, dict) and "model" in checkpoint:
    source_state = checkpoint["model"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    source_state = checkpoint["state_dict"]
else:
    source_state = checkpoint

target_state = model.state_dict()
compatible = {
    key: value
    for key, value in source_state.items()
    if key in target_state and target_state[key].shape == value.shape
}

result = model.load_state_dict(compatible, strict=False)

print("\nCheckpoint audit")
print("Source tensors:", len(source_state))
print("Compatible tensors loaded:", len(compatible))
print("Target model tensors:", len(target_state))
print("Missing target keys:", len(result.missing_keys))
print("Unexpected keys:", len(result.unexpected_keys))
print("Sample missing keys:", result.missing_keys[:15])

# Verify optimizer coverage without performing an optimizer step.
trainable = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad
]
optimizer = torch.optim.AdamW(trainable, lr=cfg.lr)

model_parameter_ids = {id(p) for p in trainable}
optimizer_parameter_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

missing_from_optimizer = model_parameter_ids - optimizer_parameter_ids
duplicate_count = sum(
    len(group["params"]) for group in optimizer.param_groups
) - len(optimizer_parameter_ids)

print("\nOptimizer audit")
print("Trainable parameter tensors:", len(trainable))
print("Optimizer parameter tensors:", len(optimizer_parameter_ids))
print("Missing from optimizer:", len(missing_from_optimizer))
print("Duplicate optimizer entries:", duplicate_count)

assert len(compatible) > 0, "No checkpoint weights matched."
assert not result.unexpected_keys, result.unexpected_keys
assert not missing_from_optimizer, "Trainable parameters missing from optimizer."
assert duplicate_count == 0, "Duplicate optimizer entries detected."

print("\nSCR MODEL AND BASIC OPTIMIZER AUDIT PASSED")
print("No optimizer step performed; no checkpoint saved.")

del optimizer, model, checkpoint, source_state, target_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()


23 18:05:28 Using backbone: Segformer-B2


Device: cuda
Backbone: mit_b2_scr
SCR output directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr


23 18:05:29 Using MLP Decoder
23 18:05:29 Loading pretrained model: /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
23 18:05:29 Load model, Time usage:
	IO: 0.0792698860168457, initialize parameters: 0.03256511688232422
23 18:05:29 Initing weights ...


SCR model constructed.

Checkpoint audit
Source tensors: 869
Compatible tensors loaded: 837
Target model tensors: 837
Missing target keys: 0
Unexpected keys: 0
Sample missing keys: []

Optimizer audit
Trainable parameter tensors: 810
Optimizer parameter tensors: 810
Missing from optimizer: 0
Duplicate optimizer entries: 0

SCR MODEL AND BASIC OPTIMIZER AUDIT PASSED
No optimizer step performed; no checkpoint saved.


In [17]:

from pathlib import Path
import os

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
TRAIN_FILE = ROOT / "train.py"
CONFIG_FILE = ROOT / "config.py"

E5_CHECKPOINT = (
    ROOT / "log_TDD_mit_b2_e5" / "checkpoint" / "epoch-100.pth"
)
SCR_CHECKPOINT_DIR = ROOT / "log_TDD_mit_b2_scr" / "checkpoint"

assert TRAIN_FILE.is_file(), "train.py not found"
assert CONFIG_FILE.is_file(), "config.py not found"
assert E5_CHECKPOINT.is_file(), "SCR initialization checkpoint missing"

train_text = TRAIN_FILE.read_text()
config_text = CONFIG_FILE.read_text()

print("SCR backbone configured:", "C.backbone = 'mit_b2_scr'" in config_text)
print("SCR training initialization reference:", "mit_b2_scr" in train_text)
print("E5 initialization checkpoint exists:", E5_CHECKPOINT.exists())
print("SCR checkpoint directory:", SCR_CHECKPOINT_DIR)
print("SCR output directory differs from E5:", 
      "log_TDD_mit_b2_scr" != "log_TDD_mit_b2_e5")

print("\nRelevant training-script lines:")
for line_no, line in enumerate(train_text.splitlines(), start=1):
    if any(term in line.lower() for term in [
        "mit_b2_scr", "epoch-100.pth", "load_state_dict",
        "optimizer", "checkpoint_dir"
    ]):
        print(f"{line_no}: {line.strip()}")

print("\nNo training started and no checkpoint was modified.")


SCR backbone configured: True
SCR training initialization reference: True
E5 initialization checkpoint exists: True
SCR checkpoint directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint
SCR output directory differs from E5: True

Relevant training-script lines:
62: # Do not restore E2 optimizer, epoch, or training state.
66: e2_checkpoint_path = r"/home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth"
80: result = model.load_state_dict(compatible_state, strict=False)
103: # Do not restore the E3 optimizer, epoch, or training state.
109: "log_TDD_mit_b2_e3/checkpoint/epoch-100.pth"
126: result = model.load_state_dict(
162: # Do not restore E5 optimizer, epoch, or training state.
163: if config.backbone == 'mit_b2_scr':
168: "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
194: result = model.load_state_dict(compatible_state, strict=False)
204: # Start a fresh E5 optimizer and training schedule.
208: "log_TDD_mit_b2_e4/checkpoint/epoch-100.pth"
224: re

In [18]:

from pathlib import Path

root = Path("/home/cse-sdpl/Downloads/CMX_SCR")
train_file = root / "train.py"

lines = train_file.read_text().splitlines()

# Display initialization and optimizer construction logic.
start_line = 45
end_line = 310

for i in range(start_line - 1, min(end_line, len(lines))):
    print(f"{i + 1:03d}: {lines[i]}")


045:     if (engine.distributed and (engine.local_rank == 0)) or (not engine.distributed):
046:         tb_dir = config.tb_dir + '/{}'.format(time.strftime("%b%d_%d-%H-%M", time.localtime()))
047:         generate_tb_dir = config.tb_dir + '/tb'
048:         tb = SummaryWriter(log_dir=tb_dir)
049:         engine.link_tb(tb_dir, generate_tb_dir)
050: 
051:     # config network and criterion
052:     criterion = nn.CrossEntropyLoss(reduction='mean', ignore_index=config.background)
053: 
054:     if engine.distributed:
055:         BatchNorm2d = nn.SyncBatchNorm
056:     else:
057:         BatchNorm2d = nn.BatchNorm2d
058:     
059:     model=segmodel(cfg=config, criterion=criterion, norm_layer=BatchNorm2d)
060: 
061:     # E3: initialize compatible model weights from E2 only.
062:     # Do not restore E2 optimizer, epoch, or training state.
063:     if config.backbone == 'mit_b2_e3':
064:         import torch
065: 
066:         e2_checkpoint_path = r"/home/cse-sdpl/Downloads/CMX_E2/log_TD

In [19]:

from pathlib import Path
import py_compile
import shutil

root = Path("/home/cse-sdpl/Downloads/CMX_SCR")
train_file = root / "train.py"

source = train_file.read_text()

old = '''        assert not result.unexpected_keys, result.unexpected_keys

    # E5: initialize model weights from the trained E4 checkpoint.'''

new = '''        assert not result.unexpected_keys, result.unexpected_keys
        assert not result.missing_keys, (
            "SCR checkpoint initialization left model keys missing: "
            f"{result.missing_keys}"
        )
        print("SCR checkpoint compatibility: PASS (no missing or unexpected keys)")

    # E5: initialize model weights from the trained E4 checkpoint.'''

if old not in source:
    raise RuntimeError(
        "Expected SCR initialization block was not found exactly. "
        "No file changes were made."
    )

backup = root / "train.py.before_scr_strict_checkpoint_audit"
if not backup.exists():
    shutil.copy2(train_file, backup)

train_file.write_text(source.replace(old, new, 1))
py_compile.compile(str(train_file), doraise=True)

print("Backup:", backup)
print("SCR strict checkpoint compatibility assertion added.")
print("Syntax check: PASS")
print("Training started: NO")


Backup: /home/cse-sdpl/Downloads/CMX_SCR/train.py.before_scr_strict_checkpoint_audit
SCR strict checkpoint compatibility assertion added.
Syntax check: PASS
Training started: NO


In [20]:

from pathlib import Path
import os
import sys
import torch

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
SCR_OUTPUT = ROOT / "log_TDD_mit_b2_scr"
TRAIN_FILE = ROOT / "train.py"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload configuration so notebook-cached settings cannot interfere.
import importlib
import config
importlib.reload(config)

cfg = config.C

assert cfg.backbone == "mit_b2_scr", cfg.backbone
assert E5_CHECKPOINT.is_file(), f"Missing E5 checkpoint: {E5_CHECKPOINT}"
assert TRAIN_FILE.is_file(), f"Missing training script: {TRAIN_FILE}"
assert "assert not result.missing_keys" in TRAIN_FILE.read_text()

print("Backbone:", cfg.backbone)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("E5 initialization checkpoint:", E5_CHECKPOINT)
print("SCR output directory:", cfg.log_dir)
print("Configured epochs:", cfg.nepochs)
print("Configured batch size:", cfg.batch_size)
print("Configured learning rate:", cfg.lr)
print("E5 checkpoint size (GB):", round(E5_CHECKPOINT.stat().st_size / 1e9, 3))

assert Path(cfg.log_dir).resolve() == SCR_OUTPUT.resolve(), (
    f"Unexpected output directory: {cfg.log_dir}"
)

print("\nPREFLIGHT CONFIGURATION CHECKS PASSED")
print("No training started; no checkpoint saved.")


Backbone: mit_b2_scr
Device: cuda
E5 initialization checkpoint: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
SCR output directory: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr
Configured epochs: 100
Configured batch size: 2
Configured learning rate: 6e-05
E5 checkpoint size (GB): 0.802

PREFLIGHT CONFIGURATION CHECKS PASSED
No training started; no checkpoint saved.


In [21]:

from pathlib import Path
import os
import sys
import importlib
import torch

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
E5_CHECKPOINT = ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Use the current SCR configuration.
import config
importlib.reload(config)
cfg = config.C

# Clear cached project modules so the current SCR code is imported.
for name in list(sys.modules):
    if name == "models" or name.startswith("models."):
        del sys.modules[name]

from models.builder import EncoderDecoder

assert cfg.backbone == "mit_b2_scr"
assert E5_CHECKPOINT.is_file()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EncoderDecoder(cfg=cfg).to(device)

checkpoint = torch.load(
    E5_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)
source_state = (
    checkpoint["model"]
    if isinstance(checkpoint, dict) and "model" in checkpoint
    else checkpoint
)

target_state = model.state_dict()
compatible_state = {
    key: value
    for key, value in source_state.items()
    if key in target_state and target_state[key].shape == value.shape
}

result = model.load_state_dict(compatible_state, strict=False)

print("Compatible checkpoint tensors:", len(compatible_state))
print("Target model tensors:", len(target_state))
print("Missing keys:", len(result.missing_keys))
print("Unexpected keys:", len(result.unexpected_keys))

assert not result.missing_keys, result.missing_keys
assert not result.unexpected_keys, result.unexpected_keys

scr_params = [
    (name, param)
    for name, param in model.named_parameters()
    if "FFMs" in name and param.requires_grad
]

print("Trainable fusion parameter tensors:", len(scr_params))
assert scr_params, "No trainable fusion parameters found"

print("\nSCR RUNTIME INITIALIZATION: PASS")
print("No optimizer step performed; no checkpoint saved.")

del model, checkpoint, source_state, target_state, compatible_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()


23 18:08:27 Using SCR-CMX: Spatial Reliability-Conditioned Fusion
23 18:08:28 Using MLP Decoder
23 18:08:28 Loading pretrained model: /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
23 18:08:28 Load model, Time usage:
	IO: 0.03600931167602539, initialize parameters: 0.02924036979675293
23 18:08:28 Initing weights ...


Compatible checkpoint tensors: 853
Target model tensors: 853
Missing keys: 0
Unexpected keys: 0
Trainable fusion parameter tensors: 100

SCR RUNTIME INITIALIZATION: PASS
No optimizer step performed; no checkpoint saved.


In [ ]:

from pathlib import Path
import os
import subprocess
import sys

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
LOG_DIR = ROOT / "log_TDD_mit_b2_scr"
CHECKPOINT_DIR = LOG_DIR / "checkpoint"

assert (ROOT / "train.py").is_file()
assert (ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth").is_file()

# Protect an existing SCR experiment from accidental overwrite.
if CHECKPOINT_DIR.exists() and any(CHECKPOINT_DIR.iterdir()):
    raise RuntimeError(
        f"SCR checkpoint directory is not empty: {CHECKPOINT_DIR}\n"
        "Inspect or back up the existing run before proceeding."
    )

os.chdir(ROOT)

print("Launching SCR-CMX training...")
print("Workspace:", ROOT)
print("Expected output:", CHECKPOINT_DIR)

result = subprocess.run(
    [
        "conda", "run", "--no-capture-output",
        "-n", "cmx",
        "python", "train.py", "-d", "0"
    ],
    cwd=str(ROOT),
    check=False
)

print("Training process exit code:", result.returncode)


Launching SCR-CMX training...
Workspace: /home/cse-sdpl/Downloads/CMX_SCR
Expected output: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_scr/checkpoint


23 18:09:14 PyTorch Version 2.14.1+cu130
23 18:09:14 using devices 0
23 18:09:15 Using SCR-CMX: Spatial Reliability-Conditioned Fusion
/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
23 18:09:17 Using MLP Decoder
23 18:09:17 Loading pretrained model: /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
23 18:09:17 Load model, Time usage:
	IO: 0.06279850006103516, initialize parameters: 0.02893972396850586
23 18:09:17 Initing weights ...
23 18:09:18 begin trainning:


SCR initialized from E5: /home/cse-sdpl/Downloads/CMX_SCR/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
Compatible checkpoint tensors: 853
Missing model keys: 0
Unexpected checkpoint keys: 0
Sample missing keys: []
SCR checkpoint compatibility: PASS (no missing or unexpected keys)
Trainable tensors: 826
Optimizer tensors: 826
Reliability-head tensors: 16
Parameters missing from optimizer: []
Duplicate optimizer entries: 0
mit_b2_scr optimizer audit: PASS
[00:00<?,?it/s]

/home/cse-sdpl/Downloads/CMX_SCR/train.py:380: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  + ' loss=%.4f total_loss=%.4f' % (loss, (sum_loss / (idx + 1)))


Epoch 1/100 Iter 612/785: lr=4.6701e-06 loss=0.0826 total_loss=0.0779: [08:26<02:23, 1.21it/s]